In [1]:
import json
import datetime as dt
import pandas as pd

from dateutil.relativedelta import relativedelta

from rockyclickup.wrapper import Session as rcu_session
from rockyclickup.utils import response_to_dataframe as rcu_res_to_df

from rockyelevate.wrapper import Session as elv_session
from rockyelevate.utils import response_to_dataframe as elv_res_to_df

In [2]:
elv = elv_session("PROD", multithread=True, max_threads=40) 

In [3]:
# today = dt.datetime.now()
today = dt.datetime(2026, 1, 29)
month_to_roll = dt.datetime(2026, 3, 1)

In [4]:
def get_all_organizations(refresh=False):
    filename = f"{today.strftime("%y%m%d")}_all_orgs.json"
    try:
        if refresh:
            raise FileNotFoundError("x")

        with open(filename, 'r') as f:
            all_orgs = json.load(f)

    except FileNotFoundError as e:
        all_orgs = elv.get_organizations(types=["SYSTEM", "PARTNER", "DISTRIBUTOR", "COMPANY", "SUBSIDIARY", "SUBGROUP"])

        with open(filename, 'w') as f:
            json.dump(all_orgs, f)

    return all_orgs

In [5]:
all_orgs = get_all_organizations()

all_orgs_df = elv_res_to_df(all_orgs)
filtered_org_df = all_orgs_df.copy()

In [6]:
# filter out subsidiaries
filtered_org_df = filtered_org_df[filtered_org_df['parent_id'] == 5012]

# filter out termmed organizations
filtered_org_df = filtered_org_df[filtered_org_df['organization_status_type'].isin(['ACTIVE'])]

print(len(filtered_org_df))

1065


In [7]:
def get_simple_plans(org_ids, refresh=False):
    filename = f"{today.strftime("%y%m%d")}_simple_plans.json"
    try:
        if refresh:
            raise FileNotFoundError("x")

        with open(filename, 'r') as f:
            simple_plans = json.load(f)

    except FileNotFoundError as e:
        simple_plans = elv.get_plans_by_org(oids=org_ids, detail=False)

        with open(filename, 'w') as f:
            json.dump(simple_plans, f)

    return simple_plans

In [8]:
org_ids = [int(o) for o in  filtered_org_df['id'].unique()]

simple_plans = get_simple_plans(org_ids)

simple_plan_df = elv_res_to_df(simple_plans)

for col in ['plan_year.valid_from', 'plan_year.valid_to']:
    simple_plan_df[col] = pd.to_datetime(simple_plan_df[col])

filtered_simple_plans = simple_plan_df.copy()

In [9]:

def is_last_day_of_month(date: dt.datetime):
    next_day = date + relativedelta(days=1)
    return next_day.month != date.month


def find_plans_to_roll(simple_plan_df: pd.DataFrame, month_to_roll: dt.datetime = dt.datetime(2026, 4, 1)):
    df = simple_plan_df.copy()

    # filter out short plan years (and extra long ones)
    df['plan_length'] = df.apply(lambda row: (row['plan_year.valid_to'] - row['plan_year.valid_from']).days, axis=1)
    df = df[
        (df['plan_length'] > 360) &
        (df['plan_length'] <= 366)
    ]

    # filter out plans that don't start on the first day of the month
    df = df[df['plan_year.valid_from'].apply(lambda x: x.day == 1)]

    # filter out plans that don't end on the last day of the month
    df = df[df['plan_year.valid_to'].apply(lambda x: is_last_day_of_month(x))]

    # narrow to plans that are ending the day before `month_to_roll` datetime parameter
    df = df[df['plan_year.valid_to'].apply(lambda x: (month_to_roll - x).days == 1)]

    # return filtered df
    df.reset_index(drop=True, inplace=True)
    return df


In [10]:
plans_to_roll = find_plans_to_roll(filtered_simple_plans, month_to_roll)
len(plans_to_roll)

14

In [11]:
def get_detailed_plans(plan_ids, refresh: bool = True):
    filename = f"{today.strftime("%y%m%d")}_detailed_plans.json"
    try:
        if refresh:
            raise FileNotFoundError("x")

        with open(filename, "r") as f:
            detailed_plans = json.load(f)

    except FileNotFoundError as e:
        detailed_plans = elv.get_plans_by_id(plan_ids)

        with open(filename, "w") as f:
            json.dump(detailed_plans, f)

    return detailed_plans

In [12]:
simple_plan_ids = [int(i) for i in plans_to_roll['id'].unique()]

detailed_plans = get_detailed_plans(simple_plan_ids)

detailed_plans_df = elv_res_to_df(detailed_plans)

for col in ['plan_year.valid_from', 'plan_year.valid_to']:
    detailed_plans_df[col] = pd.to_datetime(detailed_plans_df[col])

filtered_detailed_plans = detailed_plans_df.copy()

In [13]:
def find_rolled_plan(plans_to_roll, all_plans):
    out_df = plans_to_roll.copy()
    out_df['rolled_plan_id'] = None
    out_df['account_type'] = out_df['account_type.account_type']


    merge_df = out_df.merge(
        all_plans[['id', 'organization_id', 'account_type', 'plan_year.valid_from']],
        on=['organization_id', 'account_type'],
        how='left',
        suffixes=('', '_rolled')
    )

    merge_df = merge_df[merge_df['plan_year.valid_from'] > merge_df['plan_year.valid_to']]

    rolled_plans = merge_df.groupby(merge_df.index)['id'].apply(list).to_dict()

    out_df['rolled_plan_id'] = out_df.index.map(rolled_plans)

    return out_df


In [14]:
plans_rolled_check = find_rolled_plan(filtered_detailed_plans, simple_plan_df)

plans_not_rolled = plans_rolled_check[plans_rolled_check['rolled_plan_id'].isna()].sort_values("organization_id")

In [15]:
plans_not_rolled.to_pickle(f"{month_to_roll.strftime("%B%y").lower()}_plans_to_roll.pkl")